This notebook is literally just being used to change the avg_vol module so that it has more flexible path-finding structures. Ignore it.

modularizing avg_vol: need to make it take files from any paths (i.e. paths not hard-coded in...)

Our main thing is getting the weeks loop, so we can make a function (inside the .py file) that looks through the directory for the subject's folder (which either contains weeks as subfolders or week files)

So to deal with eitheir having weeks as files or weeks in subfolders, we can just have the input to avg_vol be the path...?

With the reference file: we can use its parent, but sometimes the files come with week--> subj, so we would need to go up two parents. That doesn't really work.

It should look in base_dir for the file. So need to make a separate function to look for the files.

In [ ]:
from pathlib import Path

base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'

def parent_lookup(file_path, subj_id):
    """
    Looks for subject parent directory of a reference file. Also tracks how many *n* parents there are.

    e.g. If you have a reference file and want to access all other files in its parent (or grandparent, n^th-level parent) directory.

    Inputs:
        file_path (str or Posix path)
        subj_id: str

    Output: subj_id parent folder
    """

    path = Path(file_path)

    for level, parent in enumerate(path.parents, start = 1):
        if parent.name == subj_id:
            return parent, level
    
    print(f'{subj_id} not found in {path}. byeeee')
    return None

In [ ]:
subj_id = 'CU_2538'
refT1 = 'W0'
#ref_img = f'{base_dir}/MNISym_GM/{subj_id}/{subj_id}_{refT1}_reslice.nii.gz'
ref_img = f'{base_dir}/anatomicals/{subj_id}/{refT1}/c2{subj_id}_{refT1}_T1.nii'

In [56]:
trial_path, level = parent_lookup(file_path = ref_img, subj_id = subj_id)

In [57]:
trial_path, level

(PosixPath('/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538'), 2)

In [ ]:
def find_week_files(file_path, subj_id, file_naming, subdir = None):
    """
    Finds files in a directory

    If subdir is not None, then looks for structure subj_id/subdir, and searches for week files within subdir

    Inputs:
        file_path (str): reference file
        subj_id (str)
        file_naming (str): naming convention of files being searched for
            e.g. if looking for all .nii files, do *.nii; if looking for wm seg files, do c2*.nii
        subdir (str): default is None; subdirectory (within subject's folder) to look inside for week folder/file

    Return:
        PosixPaths (string tuple)
    """
    parent, level = parent_lookup(file_path, subj_id)

    # folder to search inside: subj_id/subdir/ or subj_id/
    folder_root = parent/subdir if subdir else parent

    # depth of file
    rel_parts = Path(file_path).relative_to(folder_root).parts # get all components of file path
    depth = len(rel_parts)

    # then goes down *n* folders to find the file
    glob_pattern = "/".join(["*"] * (depth - 1) + [file_naming])

    # return file (if found) as PosixPath
    return [p for p in folder_root.glob(glob_pattern) if p.is_file()]

In [62]:
weeks_temp_list = find_week_files(ref_img, subj_id, file_naming = "c2*.nii")


In [64]:
trial_path, level = parent_lookup(file_path = ref_img, subj_id = subj_id)

In [66]:
trial_path

# so we can take trial_path part and then have an input that's subdir and weeks, and for whatever combination of those are inserted, it looks for files within that folder.
# so instead of the base_path, week_path part, we have this for the base_path, 

# depending on how many levels this returns, it goes back down that many levels and then looks for file with {week}

PosixPath('/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538')

**building file path**

So the path will look like:
base_path = (trial_path/subdir if subdir not None else trial_path)


week_folder = (True/False, boolean)

suffix: (str)

And look for tissue segmentation, if requested:

prefix = '' # prefix is nothing by default; only has something if there is a tissue segmentation file to be found

if not tissue=None:
    prefix = tissue_dict[tissue] + '_'

Then: if week_folder not False, for week in weeks, week_path = f'{base_path}/{week}/{prefix}{subj_id}_{week}_{suffix}.nii.gz

else week_path = f'{base_path}/{prefix}{subj_id}__{week}_{suffix}.nii.gz

In [83]:
"""
Inputs:
    reference_image: in avg_vol
    subj_id: in avg_vol

    subdir: in avg_vol (NEED TO CHANGE NAME HERE)
    
    suffix: in avg_vol; DOCUMENTATION: NEED TO INCLUDE .NII OR .NII.GZ
    tissue: in avg_vol

    week_folder: need to add to avg_vol; boolean; whether files are stored within a weeks folder or just flat (then False)
"""

parent_path, level = parent_lookup(file_path = reference_image, subj_id = subj_id)

base_path = (Path(parent_path/subdir) if not subdir == None else parent_path)
# need to check both instances of base_path

prefix = '' # empty string by default

if not tissue == None:
    #prefix = tissue_dict[tissue] + '_'
    print(tissue)


# this part is the weeks loop___________________

weeks = np.array([0,4,12,24,52])

for week in weeks:
    if week_folder == True:
        week_path = f'{base_path}/W{week}/{prefix}{subj_id}_W{week}_{suffix}'
    else:
        week_path = f'{base_path}/{prefix}{subj_id}_W{week}_{suffix}'

    # check if file exists
    if not os.path.exists(week_path):
        # continue
        print("skipping", subj_id, week)

skipping CU_2538 12
skipping CU_2538 24
skipping CU_2538 52


In [84]:
import numpy as np
import os

In [85]:
reference_image = ref_img

subdir = None

tissue = None

week_folder = True

suffix = 'T1.nii'

In [86]:
week_path

'/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W52/CU_2538_W52_T1.nii'

In [63]:
Path(weeks_temp_list[0]).parts

('/',
 'cifs',
 'diedrichsen',
 'data',
 'smarts_cerebellum',
 'anatomicals',
 'CU_2538',
 'W0',
 'c2CU_2538_W0_T1.nii')

## sufficient_weeks loop edit with week_path_builder fcn

For `sufficient_weeks` loop: we should use the path from `parent_lookup` as the path to look for files in.

It needs to iterate through the weeks, so look for week folders or week files.

So perhaps check all levels?

We can track how many levels *up* it went (from reference image) to find the subject-parent folder and go down that many levels to find the file for each week. So we would need a week-loop in there. Perhaps we can literally just have a list of all *potential* weeks and it'll look for the file for each of those weeks - if it doesn't exist, skip.

**For the sufficient_weeks loop:**

This function will be called with-in avg_vol, which is called in a subj_unique loop.

So from the loop, we have input:

- subj_id

- ref_image

- tissue (if not tissue == None, then add tissue_dict[tissue] to file_naming as input)

In [ ]:
from helper_functions import week_path_builder

In [ ]:

this_path, all_weeks = week_path_builder.week_path_builder(reference_image = ref_img, 
                              subj_id = subj_id,
                              subdir = None,
                              suffix = 'MNISym_GM_reslice.nii.gz',
                              tissue = None,
                              week_folder = False
                              )

Try out this function for sufficient weeks

In [ ]:
def sufficient_weeks():

    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c3'
    }

    weeks = np.array([0,4,12,24,52])

    # THIS WEEKS LOOP CAN BE REPLACED WITH OUR WEEKS_PATH_BUILDER FUNCTION.

    p_paths, p_weeks = week_path_builder.week_path_builder(
                                                            # from avg_vol input
                                                            reference_image = reference_img,
                                                            subj_id = subj_id,
                                                            subdir = subdir,
                                                            suffix = input_suffix,
                                                            tissue = tissue,
                                                            week_folder = week_folder


    )
    if len(p_weeks) == 1: # only one measurement week available
        return None # exit function (skip subject)    
    
    return p_weeks, p_paths

## response_matrix loop edit with week_path_builder fcn

In [ ]:
def response_matrix(Y, 
                    x, y, z,

                    # returned from sufficient_weeks loop
                    p_weeks, p_paths
                    ):
    
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c3'
    }

    # loop through both lists at the same time: zip()
        # (these lists SHOULD be the exact same length, both from week_path_builder)
    for week_path, week_num in zip(p_paths, p_weeks):
        week_img = nib.load(week_path)

        week_dict = {
            'W0': 0,
            'W4': 1,
            'W12': 2,
            'W24': 3,
            'W52': 4
        }

        Y[week_dict[str(week_num)],:] = nt.sample_image(week_img, # response matrix
                                xm=x, ym = y, zm = z, # world coordinates
                                interpolation = 1 # using trilinear resampling
                                ).flatten() # need to put each week as a row
        
    return Y

# IGNORE

file_naming should follow the same convention as the reference image.

We could just have the same suffix input here as we have in avg_vol.

We should also print out which files avg_vol is using in the regression.

for this part:


"""

for week in weeks:
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

        # e.g. if in cerebellar_alignment dir
        if not subpath == None:
            base_path =  f'{anat_dir}/{subj_id}/{subpath}/W{week}'
        else:
            base_path =  f'{anat_dir}/{subj_id}/W{week}'

        if not tissue==None:
            week_path = f'{base_path}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        else:
            week_path = f'{base_path}/{subj_id}_W{week}_T1.nii'

"""

We can just replace it with parent_lookup() so it'll return a string tuple of all the files that are available.

- subdirectories (e.g. the week_path part) isn't needed, because (for each subject) that's just the returned parent (PosixPath) from this function.

Then for RESPONSE_MATRIX, we just can loop through all the files returned in the find_file_weeks string tuple, load and resample. The only thing is, we'll need to find their week.

I think we can just use the parent_lookup and then have as input whether there are week_folders (so look inside the folders, add folders to path) or just week files.

As a bandage solution, we can just have a bunch of cases and inputs. Instead of doing all this.

So have base_dir as the root, and then looks for week_folders if that input is true, otherwise just looks in subj_id (or subj_id/subpath)